In [1]:
!pip install duckdb sentence-transformers rdflib numpy

In [3]:
import os
import re
import json
from typing import List, Dict, Optional, Tuple
import duckdb
import numpy as np
import rdflib
from rdflib.namespace import RDFS, SKOS, DCTERMS
from sentence_transformers import SentenceTransformer

# =====================================================================
# 1. DOMAIN ACRONYM & SYMBOL NORMALIZATION MAP
# =====================================================================
# Maps domain shorthand from the benchmark (DONES & AI Papers) to canonical DBpedia labels
DOMAIN_ALIASES: Dict[str, List[str]] = {
    "otr": ["Transition radiation"],
    "scintillation": ["Scintillation (physics)", "Scintillator", "Luminescence"],
    "be7": ["Beryllium-7"],
    "be-7": ["Beryllium-7"],
    "7be": ["Beryllium-7"],
    "tritium": ["Tritium"],
    "3h": ["Tritium"],
    "li": ["Lithium", "Lithium-6", "Lithium-7"],
    "dones": ["International Fusion Materials Irradiation Facility", "Neutron source"],
    "ifmif": ["International Fusion Materials Irradiation Facility"],
    "fcm": ["Fuzzy cognitive map", "Cognitive map"],
    "fcms": ["Fuzzy cognitive map"],
    "rq-vae": ["Variational autoencoder", "Vector quantization"],
    "rq-kmeans": ["K-means clustering", "Vector quantization"],
    "simclr": ["Self-supervised learning"],
    "lti": ["Linear time-invariant system", "State-space representation", "Convolution"],
    "mamba": ["State-space representation", "Linear time-invariant system"],
    "byzantine": ["Byzantine fault", "Federated learning", "Geometric median"],
    "gpais": ["Artificial general intelligence", "Active learning (machine learning)"],
    "banach": ["Banach space", "Normed vector space", "Dual space", "Dense set"],
    "lush": ["Banach space", "Unit sphere", "Numerical range"],
    "rope": ["Region of practical equivalence", "Bayesian inference", "Credible interval"],
}


# =====================================================================
# 2. HYBRID DUCKDB KG-RAG ENGINE
# =====================================================================

class HybridKGRetriever:
    def __init__(
        self,
        db_path: str = "kg_rag.duckdb",
        embedding_model_name: str = "BAAI/bge-small-en-v1.5",
        embedding_dim: int = 384,
    ):
        self.db_path = db_path
        self.embedding_dim = embedding_dim
        self.encoder = SentenceTransformer(embedding_model_name)
        self.conn = duckdb.connect(db_path)
        self._init_schema()

    def _init_schema(self):
        """Initializes DuckDB tables for nodes (with fixed-size FLOAT arrays), properties, and edges."""
        self.conn.execute(f"""
            CREATE TABLE IF NOT EXISTS kg_nodes (
                uri VARCHAR PRIMARY KEY,
                label VARCHAR,
                label_lower VARCHAR,
                definition VARCHAR,
                categories VARCHAR[],
                search_text VARCHAR,
                embedding FLOAT[{self.embedding_dim}]
            );
            CREATE TABLE IF NOT EXISTS kg_properties (
                uri VARCHAR,
                predicate VARCHAR,
                object_val VARCHAR
            );
            CREATE TABLE IF NOT EXISTS kg_edges (
                src_uri VARCHAR,
                relation VARCHAR,
                dst_uri VARCHAR
            );
        """)

    # -----------------------------------------------------------------
    # INGESTION: Load JSONL Node Cards & Turtle Graph into DuckDB
    # -----------------------------------------------------------------
    def build_index(
        self,
        jsonl_path: str = "kg_rag_chunks.jsonl",
        ttl_path: str = "domain_kg.ttl",
        batch_size: int = 256,
    ):
        """Populates the DuckDB vector and graph tables from the extracted DBpedia files."""
        node_count = self.conn.execute("SELECT COUNT(*) FROM kg_nodes").fetchone()[0]
        if node_count > 0:
            print(f"[HybridKGRetriever] Loaded existing index with {node_count} nodes.")
            return

        print(f"[HybridKGRetriever] Indexing nodes from {jsonl_path}...")
        records = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

        # Prepare texts for dense embedding (combining label, categories, properties, and definition)
        texts_to_embed = []
        for r in records:
            cats_str = ", ".join(r.get("categories", [])[:5])
            props_str = "; ".join(r.get("properties", [])[:8])
            txt = f"{r['label']}. Categories: {cats_str}. {props_str}. {r.get('definition', '')}"
            r["search_text"] = txt.strip()
            texts_to_embed.append(txt.strip())

        print(f"[HybridKGRetriever] Computing {self.embedding_dim}-d embeddings for {len(records)} nodes...")
        embeddings = self.encoder.encode(
            texts_to_embed,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )

        # Batch insert nodes into DuckDB
        node_rows = [
            (
                r["uri"],
                r["label"],
                r["label"].lower(),
                r.get("definition", ""),
                r.get("categories", []),
                r["search_text"],
                embeddings[idx].tolist(),
            )
            for idx, r in enumerate(records)
        ]
        self.conn.executemany(
            "INSERT OR REPLACE INTO kg_nodes VALUES (?, ?, ?, ?, ?, ?, ?)",
            node_rows,
        )

        # Parse Turtle file for explicit edges and structured property triples
        if os.path.exists(ttl_path):
            print(f"[HybridKGRetriever] Loading triples from {ttl_path} into SQL graph tables...")
            g = rdflib.Graph()
            g.parse(ttl_path, format="turtle")

            prop_rows = []
            edge_rows = []
            for s, p, o in g:
                s_str, p_str, o_str = str(s), str(p), str(o)
                p_short = p_str.split("/")[-1].split("#")[-1]

                if p in (RDFS.label, RDFS.comment):
                    continue
                elif p == DCTERMS.subject or p == SKOS.broader or "wikiPageWikiLink" in p_short:
                    edge_rows.append((s_str, p_short, o_str))
                else:
                    # Clean URI object values if applicable
                    if isinstance(o, rdflib.URIRef):
                        o_clean = o_str.split("/")[-1].replace("_", " ")
                    else:
                        o_clean = o_str
                    prop_rows.append((s_str, p_short, o_clean))

            if prop_rows:
                self.conn.executemany("INSERT INTO kg_properties VALUES (?, ?, ?)", prop_rows)
            if edge_rows:
                self.conn.executemany("INSERT INTO kg_edges VALUES (?, ?, ?)", edge_rows)

        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_nodes_lbl ON kg_nodes(label_lower);")
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_props_uri ON kg_properties(uri);")
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_edges_src ON kg_edges(src_uri);")
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_edges_dst ON kg_edges(dst_uri);")
        print("[HybridKGRetriever] Indexing complete.")

    # -----------------------------------------------------------------
    # STEP 1: Hybrid Entity Linking (Vector + Lexical + Alias Expansion)
    # -----------------------------------------------------------------
    def link_entities(self, question: str, top_k: int = 6) -> List[Dict]:
        """
        Combines dense cosine similarity with exact lexical/alias boosts
        to identify anchor entities in the question.
        """
        q_emb = self.encoder.encode(question, normalize_embeddings=True).tolist()
        q_lower = question.lower()
        tokens = set(re.findall(r"[a-z0-9\-]+", q_lower))

        # Check for domain aliases in question tokens
        alias_targets = set()
        for tok in tokens:
            if tok in DOMAIN_ALIASES:
                for target in DOMAIN_ALIASES[tok]:
                    alias_targets.add(target.lower())

        # Query top 25 candidates by dense vector similarity
        sql = f"""
            SELECT
                uri,
                label,
                definition,
                categories,
                array_cosine_similarity(embedding, ?::FLOAT[{self.embedding_dim}]) AS vec_score
            FROM kg_nodes
            ORDER BY vec_score DESC
            LIMIT 25
        """
        candidates = self.conn.execute(sql, [q_emb]).fetchall()

        # Also pull exact alias/substring matches that might rank lower in pure vector search
        if alias_targets:
            placeholders = ", ".join(["?"] * len(alias_targets))
            alias_sql = f"""
                SELECT
                    uri,
                    label,
                    definition,
                    categories,
                    array_cosine_similarity(embedding, ?::FLOAT[{self.embedding_dim}]) AS vec_score
                FROM kg_nodes
                WHERE label_lower IN ({placeholders})
            """
            alias_rows = self.conn.execute(alias_sql, [q_emb] + list(alias_targets)).fetchall()
            candidates.extend(alias_rows)

        # Deduplicate and compute hybrid score (Reciprocal/Additive Fusion)
        scored_map: Dict[str, Dict] = {}
        for uri, label, definition, categories, vec_score in candidates:
            if uri in scored_map:
                continue
            lbl_low = label.lower()
            lexical_boost = 0.0

            # Boost if the entity label appears verbatim in the question (length >= 3)
            if len(lbl_low) >= 3 and lbl_low in q_lower:
                lexical_boost += 0.35
            # Boost if triggered by our domain acronym/isotope alias map
            if lbl_low in alias_targets:
                lexical_boost += 0.40

            scored_map[uri] = {
                "uri": uri,
                "label": label,
                "definition": definition,
                "categories": categories,
                "hybrid_score": float(vec_score or 0.0) + lexical_boost,
            }

        ranked = sorted(scored_map.values(), key=lambda x: x["hybrid_score"], reverse=True)
        return ranked[:top_k]

    # -----------------------------------------------------------------
    # STEP 2: Structured Property & 1-Hop Neighborhood Retrieval
    # -----------------------------------------------------------------
    def get_node_details(self, uris: List[str]) -> Tuple[Dict[str, List[str]], Dict[str, List[str]]]:
        """Fetches structured physical/math properties and 1-hop connected concepts for seed URIs."""
        if not uris:
            return {}, {}

        placeholders = ", ".join(["?"] * len(uris))

        # 1. Fetch infobox/ontological properties (halfLife, density, etc.)
        prop_rows = self.conn.execute(
            f"SELECT uri, predicate, object_val FROM kg_properties WHERE uri IN ({placeholders})",
            uris,
        ).fetchall()

        props_by_uri: Dict[str, List[str]] = {}
        for u, pred, val in prop_rows:
            props_by_uri.setdefault(u, []).append(f"{pred}: {val}")

        # 2. Fetch 1-hop outgoing and incoming semantic neighbors
        edge_sql = f"""
            SELECT e.src_uri, n.label AS neighbor_label
            FROM kg_edges e
            JOIN kg_nodes n ON e.dst_uri = n.uri
            WHERE e.src_uri IN ({placeholders}) AND e.relation = 'wikiPageWikiLink'
            UNION
            SELECT e.dst_uri AS src_uri, n.label AS neighbor_label
            FROM kg_edges e
            JOIN kg_nodes n ON e.src_uri = n.uri
            WHERE e.dst_uri IN ({placeholders}) AND e.relation = 'wikiPageWikiLink'
        """
        edge_rows = self.conn.execute(edge_sql, uris + uris).fetchall()

        neighbors_by_uri: Dict[str, List[str]] = {}
        for u, n_label in edge_rows:
            neighbors_by_uri.setdefault(u, []).append(n_label)

        return props_by_uri, neighbors_by_uri

    # -----------------------------------------------------------------
    # STEP 3: 2-Hop Bridge & Shared Context Discovery (Comparative Qs)
    # -----------------------------------------------------------------
    def find_connecting_bridges(self, uris: List[str], max_bridges: int = 5) -> List[Dict]:
        """
        Finds intermediate bridge entities or shared categories that connect at least two
        distinct seed entities in `uris` (ideal for comparative & multi-hop questions).
        """
        if len(uris) < 2:
            return []

        placeholders = ", ".join(["?"] * len(uris))
        bridge_sql = f"""
            WITH undirected_edges AS (
                SELECT src_uri AS u1, dst_uri AS u2, relation FROM kg_edges
                UNION ALL
                SELECT dst_uri AS u1, src_uri AS u2, relation FROM kg_edges
            )
            SELECT
                b.uri AS bridge_uri,
                b.label AS bridge_label,
                b.definition AS bridge_def,
                COUNT(DISTINCT e.u1) AS connected_seeds,
                LIST(DISTINCT n_seed.label) AS linked_seed_labels
            FROM undirected_edges e
            JOIN kg_nodes b ON e.u2 = b.uri
            JOIN kg_nodes n_seed ON e.u1 = n_seed.uri
            WHERE e.u1 IN ({placeholders})
              AND e.u2 NOT IN ({placeholders})
            GROUP BY b.uri, b.label, b.definition
            HAVING COUNT(DISTINCT e.u1) >= 2
            ORDER BY connected_seeds DESC
            LIMIT {max_bridges}
        """
        rows = self.conn.execute(bridge_sql, uris + uris).fetchall()
        bridges = []
        for b_uri, b_lbl, b_def, count, seed_lbls in rows:
            bridges.append({
                "uri": b_uri,
                "label": b_lbl,
                "definition": b_def,
                "linked_seeds": seed_lbls,
            })
        return bridges

    # -----------------------------------------------------------------
    # STEP 4: Question-Type-Aware Context Assembly for LLM Prompting
    # -----------------------------------------------------------------
    def retrieve_and_format(
        self,
        question: str,
        question_type: Optional[str] = None,
        top_k: int = 5,
        max_context_chars: int = 3200,
    ) -> str:
        """
        Executes hybrid retrieval adapted to `question_type` ('comparative', 'procedural',
        'factual', 'causal', 'predictive') and formats a compact Markdown block for RAG.
        """
        q_type = (question_type or "factual").lower()
        if q_type == "comparative":
            top_k = max(top_k, 6)  # Ensure both compared entities and their sub-concepts are captured

        seed_nodes = self.link_entities(question, top_k=top_k)
        if not seed_nodes:
            return ""

        seed_uris = [n["uri"] for n in seed_nodes]
        props_map, neighbors_map = self.get_node_details(seed_uris)

        lines = ["### Domain Knowledge Graph Context (DBpedia)"]

        # 1. Format Primary Linked Entities
        for node in seed_nodes:
            uri = node["uri"]
            lbl = node["label"]
            defn = node["definition"]
            cats = ", ".join(node["categories"][:4]) if node["categories"] else "N/A"
            props = props_map.get(uri, [])
            neighbors = neighbors_map.get(uri, [])

            entry = f"- **{lbl}** (Categories: {cats}): {defn}"
            if props:
                # Prioritize quantitative properties for procedural/factual questions
                prop_limit = 10 if q_type in ("procedural", "factual") else 5
                entry += f"\n  - *Attributes & Constants*: {'; '.join(props[:prop_limit])}"
            if neighbors:
                neighbor_limit = 12 if q_type in ("causal", "predictive") else 6
                entry += f"\n  - *Related Domain Concepts*: {', '.join(neighbors[:neighbor_limit])}"
            lines.append(entry)

        # 2. Add 2-Hop Bridge Entities for Comparative / Causal Questions
        if q_type in ("comparative", "causal"):
            bridges = self.find_connecting_bridges(seed_uris, max_bridges=4)
            if bridges:
                lines.append("\n### Shared Conceptual Bridges & Intersections")
                for b in bridges:
                    linked_str = " <-> ".join(b["linked_seeds"])
                    lines.append(f"- **{b['label']}** (Connects: {linked_str}): {b['definition']}")

        context_block = "\n".join(lines)
        if len(context_block) > max_context_chars:
            context_block = context_block[:max_context_chars].rsplit("\n", 1)[0] + "\n..."
        return context_block


# =====================================================================
# 3. EXAMPLE INTEGRATION WITH BENCHMARK QUESTIONS
# =====================================================================

if __name__ == "__main__":
    retriever = HybridKGRetriever(db_path="kg_rag.duckdb")

    # Build index once from the files generated by build_dbpedia_kg.py
    if os.path.exists("kg_rag_chunks.jsonl"):
        retriever.build_index(jsonl_path="kg_rag_chunks.jsonl", ttl_path="domain_kg.ttl")

        # Sample test cases mirroring sample_qa.json
        sample_queries = [
            {
                "question": "Both OTR and scintillation are proposed for experimental beam imaging diagnostics, but they produce photons by different mechanisms. Which physical process does scintillation require that OTR does not?",
                "question_type": "comparative",
            },
            {
                "question": "What would be the radioactive decay constant of Be7 if we round its half life in days to the nearest integer?",
                "question_type": "procedural",
            },
            {
                "question": "In the paper of Mamba, why Linear Time Invariance models struggle with selective Copying?",
                "question_type": "causal",
            },
        ]

        for q in sample_queries:
            print("=" * 80)
            print(f"QUESTION ({q['question_type']}): {q['question']}\n")
            kg_context = retriever.retrieve_and_format(
                question=q["question"],
                question_type=q["question_type"],
            )
            print(kg_context)
    else:
        print("Run `build_dbpedia_kg.py` first to generate `kg_rag_chunks.jsonl` and `domain_kg.ttl`.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[HybridKGRetriever] Indexing nodes from kg_rag_chunks.jsonl...
[HybridKGRetriever] Computing 384-d embeddings for 8342 nodes...


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

[HybridKGRetriever] Loading triples from domain_kg.ttl into SQL graph tables...
[HybridKGRetriever] Indexing complete.
QUESTION (comparative): Both OTR and scintillation are proposed for experimental beam imaging diagnostics, but they produce photons by different mechanisms. Which physical process does scintillation require that OTR does not?

### Domain Knowledge Graph Context (DBpedia)
- **Scintillation (physics)** (Categories: N/A): 
  - *Related Domain Concepts*: Scintillator, Fluorescence, Compact Linear Collider, Scintillation counter, Atomic number, Autoradiograph
- **Scintillator** (Categories: Particle detectors): 
  - *Related Domain Concepts*: Nuclear physics, Luminescence, Potassium iodide, Gadolinium, Atomic battery, Neutrino detector
- **Transition radiation** (Categories: N/A): 
  - *Related Domain Concepts*: Dielectric, Transition radiation detector, Relativistic particle, Particle beam, Hadron
- **Luminescence** (Categories: N/A): 
  - *Related Domain Concepts*: Expone

In [4]:
def build_competition_prompt(question: str, question_type: str, doc_chunks: List[str], kg_context: str) -> str:
    docs_formatted = "\n\n".join(f"[Doc {i+1}] {c}" for i, c in enumerate(doc_chunks))
    return f"""You are a domain expert in nuclear fusion facilities (IFMIF-DONES), computational intelligence, and applied mathematics.
Answer the question accurately using the provided Scientific Literature Passages and the Domain Knowledge Graph Context.

Guidelines:
1. Prioritize specific experimental setups, equations, figures, and tables from the [Scientific Literature Passages].
2. Use the [Domain Knowledge Graph Context] to ground physical definitions, mathematical properties, isotope constants, and conceptual comparisons.
3. For procedural questions, show every step of the mathematical derivation clearly in LaTeX.

{kg_context}

### Scientific Literature Passages
{docs_formatted}

### Question (Type: {question_type})
{question}

### Expert Answer:"""

In [5]:
import os
import re
import json
from typing import List, Dict, Optional, Tuple
import duckdb
import torch
from sentence_transformers import SentenceTransformer

ACRONYM_PATTERN = re.compile(
    r"\b([A-Z][A-Za-z0-9\-\s]{3,45}?)\s*\(([A-Z0-9\-]{2,12})\)"
)


class UnifiedCompetitionRAG:
    """
    Single-file DuckDB engine storing both Literature Chunks and the DBpedia KG.
    Optimized for low disk footprint (<100 MB KG) and fast indexed subgraph queries.
    """

    def __init__(
        self,
        db_path: str = "unified_rag.duckdb",
        model_name_or_path: str = "BAAI/bge-small-en-v1.5",
        embedding_dim: int = 384,
    ):
        self.db_path = db_path
        self.embedding_dim = embedding_dim
        device = "cuda" if torch.cuda.is_available() else "cpu"
        # Load in float16 to cut memory/disk footprint by 50% (~67 MB)
        self.encoder = SentenceTransformer(
            model_name_or_path,
            device=device,
            model_kwargs={"torch_dtype": torch.float16} if device == "cuda" else {},
        )
        self.conn = duckdb.connect(db_path)
        self._init_schema()

    def _init_schema(self):
        """Compact schema without redundant search_text columns."""
        self.conn.execute(f"""
            CREATE TABLE IF NOT EXISTS kg_nodes (
                uri VARCHAR PRIMARY KEY,
                label VARCHAR,
                label_lower VARCHAR,
                definition VARCHAR,
                categories VARCHAR[],
                embedding FLOAT[{self.embedding_dim}]
            );
            CREATE TABLE IF NOT EXISTS kg_aliases (
                alias_lower VARCHAR,
                uri VARCHAR
            );
            CREATE TABLE IF NOT EXISTS kg_properties (
                uri VARCHAR,
                predicate VARCHAR,
                object_val VARCHAR
            );
            CREATE TABLE IF NOT EXISTS kg_edges (
                src_uri VARCHAR,
                relation VARCHAR,
                dst_uri VARCHAR
            );
            CREATE TABLE IF NOT EXISTS doc_chunks (
                chunk_id VARCHAR PRIMARY KEY,
                domain VARCHAR,
                source_title VARCHAR,
                content VARCHAR,
                embedding FLOAT[{self.embedding_dim}]
            );
        """)

    # -----------------------------------------------------------------
    # 1. AUTOMATED ALIAS & PAPER-ACRONYM AUGMENTATION
    # -----------------------------------------------------------------
    def register_corpus_acronyms(self, chunks: List[Dict]):
        """
        Scans scientific paper chunks for 'Full Name (ACRONYM)' patterns
        (e.g., 'Optical Transition Radiation (OTR)', 'Fuzzy Cognitive Map (FCM)')
        and links the acronym to matching DBpedia URIs in `kg_aliases`.
        """
        discovered_pairs: List[Tuple[str, str]] = []
        for ch in chunks:
            text = ch.get("content", "")
            for match in ACRONYM_PATTERN.finditer(text):
                full_name = match.group(1).strip().lower()
                acronym = match.group(2).strip().lower()
                if len(acronym) >= 2 and not acronym.isdigit():
                    discovered_pairs.append((acronym, full_name))

        if not discovered_pairs:
            return

        # Link discovered acronyms to existing kg_nodes where full_name matches label_lower
        self.conn.executemany(
            """
            INSERT INTO kg_aliases (alias_lower, uri)
            SELECT DISTINCT ?, uri
            FROM kg_nodes
            WHERE label_lower = ? OR STARTS_WITH(label_lower, ?)
            """,
            [(acr, fname, fname) for acr, fname in set(discovered_pairs)],
        )
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_alias ON kg_aliases(alias_lower);")

    # -----------------------------------------------------------------
    # 2. FAST INDEXED entity LINKING (Vector + Alias Table Lookup)
    # -----------------------------------------------------------------
    def link_entities(self, question: str, top_k: int = 6) -> List[Dict]:
        q_emb = self.encoder.encode(question, normalize_embeddings=True).tolist()
        q_lower = question.lower()
        tokens = list(set(re.findall(r"[a-z0-9\-]{2,}", q_lower)))

        # 1. Top dense vector matches
        vec_sql = f"""
            SELECT
                uri, label, definition, categories,
                array_cosine_similarity(embedding, ?::FLOAT[{self.embedding_dim}]) AS vec_score
            FROM kg_nodes
            ORDER BY vec_score DESC
            LIMIT 20
        """
        candidates = self.conn.execute(vec_sql, [q_emb]).fetchall()

        # 2. Exact token / symbol / acronym lookup via `kg_aliases` and `label_lower`
        alias_uris = set()
        if tokens:
            placeholders = ", ".join(["?"] * len(tokens))
            alias_sql = f"""
                SELECT DISTINCT n.uri, n.label, n.definition, n.categories,
                       array_cosine_similarity(n.embedding, ?::FLOAT[{self.embedding_dim}]) AS vec_score
                FROM kg_nodes n
                LEFT JOIN kg_aliases a ON n.uri = a.uri
                WHERE n.label_lower IN ({placeholders})
                   OR a.alias_lower IN ({placeholders})
            """
            alias_rows = self.conn.execute(
                alias_sql, [q_emb] + tokens + tokens
            ).fetchall()
            for r in alias_rows:
                alias_uris.add(r[0])
            candidates.extend(alias_rows)

        scored: Dict[str, Dict] = {}
        for uri, label, definition, categories, vec_score in candidates:
            if uri in scored:
                continue
            boost = 0.0
            lbl_low = label.lower()
            if len(lbl_low) >= 3 and lbl_low in q_lower:
                boost += 0.35
            if uri in alias_uris:
                boost += 0.40

            scored[uri] = {
                "uri": uri,
                "label": label,
                "definition": definition,
                "categories": categories,
                "score": float(vec_score or 0.0) + boost,
            }

        return sorted(scored.values(), key=lambda x: x["score"], reverse=True)[:top_k]

    # -----------------------------------------------------------------
    # 3. OPTIMIZED 2-HOP BRIDGE QUERY (Predicate Pushdown)
    # -----------------------------------------------------------------
    def find_connecting_bridges(self, uris: List[str], max_bridges: int = 4) -> List[Dict]:
        """
        Pushes the `IN (...)` filter inside the UNION ALL branches so DuckDB only
        scans edges incident to the seed URIs rather than materializing the full graph.
        """
        if len(uris) < 2:
            return []

        placeholders = ", ".join(["?"] * len(uris))
        bridge_sql = f"""
            WITH seed_incident_edges AS (
                SELECT src_uri AS seed_uri, dst_uri AS neighbor_uri
                FROM kg_edges
                WHERE src_uri IN ({placeholders})
                  AND dst_uri NOT IN ({placeholders})
                UNION ALL
                SELECT dst_uri AS seed_uri, src_uri AS neighbor_uri
                FROM kg_edges
                WHERE dst_uri IN ({placeholders})
                  AND src_uri NOT IN ({placeholders})
            )
            SELECT
                b.uri,
                b.label,
                b.definition,
                COUNT(DISTINCT e.seed_uri) AS connected_seeds,
                LIST(DISTINCT s.label) AS linked_seed_labels
            FROM seed_incident_edges e
            JOIN kg_nodes b ON e.neighbor_uri = b.uri
            JOIN kg_nodes s ON e.seed_uri = s.uri
            GROUP BY b.uri, b.label, b.definition
            HAVING COUNT(DISTINCT e.seed_uri) >= 2
            ORDER BY connected_seeds DESC
            LIMIT {max_bridges}
        """
        params = uris + uris + uris + uris
        rows = self.conn.execute(bridge_sql, params).fetchall()
        return [
            {
                "uri": r[0],
                "label": r[1],
                "definition": r[2],
                "linked_seeds": r[4],
            }
            for r in rows
        ]

    # -----------------------------------------------------------------
    # 4. UNIFIED LITERATURE + KG RETRIEVAL
    # -----------------------------------------------------------------
    def retrieve_literature_chunks(
        self, question: str, domain: Optional[str] = None, top_k: int = 5
    ) -> List[Dict]:
        """Retrieves top paper passages from the same DuckDB database."""
        q_emb = self.encoder.encode(question, normalize_embeddings=True).tolist()
        if domain:
            sql = f"""
                SELECT chunk_id, source_title, content,
                       array_cosine_similarity(embedding, ?::FLOAT[{self.embedding_dim}]) AS score
                FROM doc_chunks
                WHERE domain = ?
                ORDER BY score DESC
                LIMIT {top_k}
            """
            rows = self.conn.execute(sql, [q_emb, domain]).fetchall()
        else:
            sql = f"""
                SELECT chunk_id, source_title, content,
                       array_cosine_similarity(embedding, ?::FLOAT[{self.embedding_dim}]) AS score
                FROM doc_chunks
                ORDER BY score DESC
                LIMIT {top_k}
            """
            rows = self.conn.execute(sql, [q_emb]).fetchall()

        return [
            {"chunk_id": r[0], "source": r[1], "content": r[2], "score": r[3]}
            for r in rows
        ]

    # -----------------------------------------------------------------
    # 5. SUBMISSION PACKAGING & DISK COMPACTION
    # -----------------------------------------------------------------
    def compact_and_clean_for_submission(self, intermediate_files: List[str]):
        """
        Reclaims unused pages in DuckDB and removes raw .ttl/.jsonl/.sqlite
        build artifacts so your final submission archive stays well under 1 GB.
        """
        self.conn.execute("CHECKPOINT;")
        self.conn.execute("VACUUM;")
        db_size_mb = os.path.getsize(self.db_path) / (1024 * 1024)
        print(f"[Packaging] Final Unified DuckDB size: {db_size_mb:.2f} MB")

        for fpath in intermediate_files:
            if os.path.exists(fpath):
                os.remove(fpath)
                print(f"[Packaging] Removed build artifact: {fpath}")